In [1]:
# 임시 코드
from pydantic import BaseModel
from typing_extensions import Annotated, Sequence, TypedDict
from langchain.schema import BaseMessage

# 헤더 모델 정의
class Headers(TypedDict):
    Authorization: []
    Content_Type: []
    
class Vehicle(TypedDict):
    brand: [] = ""  # 기본값 설정
    model: [] = ""
    date: int = 0
    year: [] = ""
    detail: [] = ""
    option: [] = ""

class State(TypedDict):
    headers : Headers
    vehicle: Vehicle
    messages: Annotated[Sequence[BaseMessage], "add_messages"]
    ask_human: bool


class HumanRequest(BaseModel):
    """Forward the conversation to an expert. Use when you can't assist directly or the user needs assistance that exceeds your authority.
    To use this function, pass the user's 'request' so that an expert can provide appropriate guidance.
    """

    request: str

In [2]:
from RAG.modules import logging

logging.langsmith("Model_RAG")

# %%
from langgraph.graph import StateGraph, START, END
from RAG.types import state
from RAG.llm.model import get_OpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain.globals import set_verbose, set_debug

set_debug(True)
set_verbose(True)

llm = get_OpenAI()
output_parser = StrOutputParser()

from RAG.tools.call_Vehicle_tools import get_brand_list_tool, get_model_list_tool, get_year_list_tool, get_detail_list_tool, get_option_list_tool, HumanRequest, State, Vehicle, Headers

tools = [get_brand_list_tool, get_model_list_tool, get_year_list_tool, get_detail_list_tool, get_option_list_tool, HumanRequest]

# LLM 모델 초기화 및 도구 바인딩
llm_with_tools = llm.bind_tools(tools)

# %%
from langgraph.graph import StateGraph, MessagesState, START, END

# %%
tool_node = ToolNode(tools)


LangSmith 추적을 시작합니다.
[프로젝트명]
Model_RAG


c:\WS\final-project\SKN03-FINAL-6Team\TailorLink_LLM\finance\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3577: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)


In [3]:
tool_node

tools(tags=None, recurse=True, func_accepts_config=True, func_accepts={'writer': False, 'store': True}, tools_by_name={'get_brand_list': StructuredTool(name='get_brand_list', description='자동차 제조사 목록을 조회하는 함수입니다.\n\nArgs:\n    headers (dict): API 호출 시 필요한 헤더 정보.\n\nReturns:\n    dict: 제조사 목록 JSON 데이터.', args_schema=<class 'langchain_core.utils.pydantic.get_brand_list'>, return_direct=True, func=<function get_brand_list_tool at 0x000001D7D5F56B60>), 'get_model_list': StructuredTool(name='get_model_list', description='자동차 모델 목록을 조회하는 함수입니다.\n\nArgs:\n    headers (dict): API 호출 시 필요한 헤더 정보.\n    brand_code (str): 브랜드 코드.\n\nReturns:\n    dict: 모델 목록 JSON 데이터.', args_schema=<class 'langchain_core.utils.pydantic.get_model_list'>, return_direct=True, func=<function get_model_list_tool at 0x000001D7D5F7D800>), 'get_year_list': StructuredTool(name='get_year_list', description='차량 연식 목록을 조회하는 함수입니다.\n\nArgs:\n    headers (dict): API 호출 시 필요한 헤더 정보.\n    brand_code (str): 브랜드 코드.\n    model_code 

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

access_token = os.getenv("CODEF_ACCESS_TOKEN")

In [5]:
# headers: Headers = {
#     "Authorization": f"Bearer {access_token}",
#     "Content-Type": "application/json"
# }

In [6]:
# State = {
#     "messages": [{"role": "user", "content": "브랜드 조회해줘."}],
#     "headers" : headers,
#     "vehicle" : Vehicle(),
#     "ask_human": False
# }

In [7]:
# Prompt 생성 함수
def call_prompt():
    return "Please provide the required information."

In [8]:
# State : State = {
#     "messages": [{"role": "user", "content": "브랜드 조회해줘."}],
#     "headers" : headers,
#     "vehicle" : Vehicle(),
#     "ask_human": False
# }

In [9]:
headers: Headers={
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

State: State = {
    "messages": [{"role": "user", "human": "브랜드 조회해줘."}],
    "headers":headers,
    "vehicle":Vehicle(),
    "ask_human": bool
}
# config 설정
config = {"configurable": {"thread_id": "1"}}

In [11]:
def call_model(State: State):
    # 메시지 처리
    user_input = State['messages']
    prompt = call_prompt()
    chain = prompt | llm_with_tools | output_parser

    # 사람에게 질문할지 여부 초기화
    ask_human = False

    # 응답 생성
    response = chain.invoke(user_input)

    # 도구 호출이 있고 이름이 'HumanRequest' 인 경우
    if response.tool_calls and response.tool_calls[0]["name"] == HumanRequest.__name__:
        ask_human = True

    
    # 메시지와 ask_human 상태 반환
    # return {"messages": [response], "ask_human": ask_human}
    return State

In [12]:
call_model(State)

TypeError: Expected a Runnable, callable or dict.Instead got an unsupported type: <class 'str'>